# AutoIntel AI – Complaint Analysis Engine
**Member 3: Ankita Manna | Branch: ankita**

This notebook demonstrates the full pipeline:
1. Generate the complaint dataset
2. Explore and preprocess the data
3. Build TF-IDF + Cosine Similarity engine
4. Test with sample complaints
5. Run evaluation

## Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Environment ready')

## Step 1: Generate Complaint Dataset

In [ ]:
from src.analysis.complaint_dataset import generate_complaint_dataset

df = generate_complaint_dataset()
print(f'Total complaints: {len(df)}')
print(f'Fault categories: {df["fault_category"].nunique()}')
df.head(10)

In [ ]:
df.groupby('fault_name')['complaint_id'].count().reset_index().rename(columns={'complaint_id': 'count'})

## Step 2: Text Preprocessing

In [ ]:
from src.analysis.text_preprocessor import preprocess

sample_complaints = [
    'My car AC is blowing warm air and not cooling the cabin at all',
    'ABS warning light is on and brakes feel spongy',
    'Engine is overheating and steam is coming from under the hood',
    'Battery keeps dying and car wont start in the morning',
    'Transmission slipping between gears when driving on highway',
]

print('Preprocessing Examples')
print('=' * 70)
for complaint in sample_complaints:
    print(f'Raw      : {complaint}')
    print(f'Processed: {preprocess(complaint)}')
    print('-' * 70)

In [ ]:
df['processed_text'] = df['complaint_text'].apply(preprocess)
df[['complaint_text', 'processed_text', 'fault_name']].head(5)

## Step 3: Build TF-IDF Matrix

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95, sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(df['processed_text'])

print(f'Matrix shape      : {tfidf_matrix.shape}')
print(f'Vocabulary size   : {len(vectorizer.vocabulary_)}')
print(f'Top 20 features   : {list(vectorizer.get_feature_names_out()[:20])}')

## Step 4: Test the Matching Engine

In [ ]:
from src.analysis.matching_engine import analyze_complaint, get_top_matches

test_queries = [
    'Engine getting very hot and I see steam from bonnet',
    'Car battery completely dead every morning',
    'Brakes are squealing loudly and pedal is soft',
    'Gearbox is slipping and shifting is delayed',
    'AC not working blowing only hot air',
    'Car misfires and engine shakes at idle',
    'Steering wheel shakes at high speed',
    'ABS light on and car skids easily',
    'Car stalling and fuel consumption is very high',
    'Black smoke from exhaust and engine running rough',
]

results = []
for query in test_queries:
    r = analyze_complaint(query)
    results.append({
        'Query': query[:55] + '...' if len(query) > 55 else query,
        'Fault': r.get('fault_name', r.get('status')),
        'OBD Code': r.get('obd_code', '-'),
        'Score': r.get('similarity_score', 0),
        'Confidence': r.get('confidence', '-'),
    })

pd.DataFrame(results)

In [ ]:
print('Full result for one complaint:')
import json
print(json.dumps(analyze_complaint('my car engine is overheating and fan is not running'), indent=2))

In [ ]:
print('Top 5 matches for ambiguous complaint:')
matches = get_top_matches('car is not working properly', top_n=5)
for m in matches:
    print(f"Rank {m['rank']}: {m['fault_name']} | Score: {m['similarity_score']} | Confidence: {m['confidence']}")

## Step 5: Evaluation

In [ ]:
from src.analysis.evaluate_engine import run_evaluation

accuracy, avg_score = run_evaluation()

## Summary

The Complaint Analysis Engine is complete and ready for Member 4 integration.

**API for Streamlit:**
```python
from src.analysis.matching_engine import analyze_complaint
result = analyze_complaint(user_complaint_text)
```

See `docs/member3/INTEGRATION_GUIDE.md` for complete integration instructions.